In [5]:
import sys
import os

# Add project root to Python path
# From notebooks directory, go up one level to get to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

In [6]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

from app.models.svi.surface_svi.phi.heston_like import HestonLikePhi, PowerLawPhi, StabilizedPowerLawPhi
from app.models.svi.surface_svi.model import SSVIFormula

In [7]:
import plotly.io as pio
pio.renderers.default = "notebook"

In [9]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
import base64

from app.models.svi.surface_svi.phi.heston_like import HestonLikePhi, PowerLawPhi, StabilizedPowerLawPhi
from app.models.svi.surface_svi.model import SSVIFormula


def build_iv_surface_for_plot(rho, sigma_atm, eta, gamma, lam, phi_model, n_k=50, n_t=50):
    """Build IV surface arrays for plotting."""
    
    # select phi
    if phi_model == "heston_like":
        phi = HestonLikePhi(lam=lam)
    elif phi_model == "power_law":
        phi = PowerLawPhi(eta=eta, gamma=gamma)
    else:
        phi = StabilizedPowerLawPhi(eta=eta, gamma=gamma)

    model      = SSVIFormula(rho=rho, phi=phi)
    k_grid     = np.linspace(-1.0, 1.0, n_k)
    T_grid     = np.linspace(0.10, 2.0, n_t)
    theta_func = lambda T: SSVIFormula.theta_of_T(T, sigma_atm=sigma_atm)

    kk, TT, w, iv = SSVIFormula.build_iv_surface(
        k_grid=k_grid,
        T_grid=T_grid,
        model=model,
        theta_func=theta_func,
    )

    # clip for display
    iv = np.clip(iv, 0.0, 1.5)

    return kk, TT, iv


def make_surface(rho, sigma_atm, eta, gamma, lam, phi_model):
    kk, TT, iv = build_iv_surface_for_plot(
        rho, sigma_atm, eta, gamma, lam, phi_model
    )

    # arbitrage check
    phi_obj = (
        HestonLikePhi(lam=lam)        if phi_model == "heston_like"   else
        PowerLawPhi(eta=eta, gamma=gamma)          if phi_model == "power_law"     else
        StabilizedPowerLawPhi(eta=eta, gamma=gamma)
    )
    model      = SSVIFormula(rho=rho, phi=phi_obj)
    theta_func = lambda T: SSVIFormula.theta_of_T(T, sigma_atm=sigma_atm)
    T_check    = np.linspace(0.10, 2.0, 20)
    theta_check = theta_func(T_check)
    cond1 = theta_check * phi_obj(theta_check) * (1 + abs(rho))
    arb_free = bool(np.all(cond1 < 4))

    return kk, TT, iv, arb_free, cond1.max()


# --- widgets ---
rho_slider = widgets.FloatSlider(
    value=-0.7, min=-0.95, max=0.95, step=0.05,
    description="ρ", continuous_update=False,
    style={"description_width": "initial"},
)
sigma_slider = widgets.FloatSlider(
    value=0.20, min=0.05, max=0.60, step=0.01,
    description="σ_ATM", continuous_update=False,
    style={"description_width": "initial"},
)
eta_slider = widgets.FloatSlider(
    value=0.5, min=0.05, max=3.0, step=0.05,
    description="η (power law)", continuous_update=False,
    style={"description_width": "initial"},
)
gamma_slider = widgets.FloatSlider(
    value=0.5, min=0.05, max=0.95, step=0.05,
    description="γ (power law)", continuous_update=False,
    style={"description_width": "initial"},
)
lam_slider = widgets.FloatSlider(
    value=2.0, min=0.1, max=10.0, step=0.1,
    description="λ (heston)", continuous_update=False,
    style={"description_width": "initial"},
)
phi_dropdown = widgets.Dropdown(
    options=["heston_like", "power_law", "stabilized_power_law"],
    value="heston_like",
    description="φ model",
    style={"description_width": "initial"},
)
arb_label = widgets.HTML(value="")

# --- initial surface ---
kk, TT, iv, arb_free, max_cond = make_surface(
    rho_slider.value, sigma_slider.value,
    eta_slider.value, gamma_slider.value,
    lam_slider.value, phi_dropdown.value,
)

# --- plotly figure ---
fig = go.FigureWidget(data=[
    go.Surface(
        x=kk, y=TT, z=iv,
        colorscale="turbo",
        colorbar=dict(title="Impl. Vol", thickness=15),
        cmin=0.0, cmax=0.80,
    )
])

fig.update_layout(
    title="SSVI Implied Volatility Surface",
    scene=dict(
        xaxis_title="Log-moneyness k",
        yaxis_title="Maturity T",
        zaxis_title="Implied Vol σ_BS",
        zaxis=dict(range=[0.0, 0.80]),
        camera=dict(eye=dict(x=-1.5, y=-1.5, z=0.8)),
    ),
    width=800,
    height=600,
    margin=dict(l=0, r=0, t=40, b=0),
)


def update_arb_label(arb_free, max_cond):
    if arb_free:
        arb_label.value = (
            f"<span style='color:green; font-weight:bold'>"
            f"✓ Arbitrage-free (max θφ(1+|ρ|) = {max_cond:.3f} &lt; 4)"
            f"</span>"
        )
    else:
        arb_label.value = (
            f"<span style='color:red; font-weight:bold'>"
            f"✗ Arbitrage violated (max θφ(1+|ρ|) = {max_cond:.3f} ≥ 4)"
            f"</span>"
        )


update_arb_label(arb_free, max_cond)


def on_change(change):
    kk, TT, iv, arb_free, max_cond = make_surface(
        rho_slider.value, sigma_slider.value,
        eta_slider.value, gamma_slider.value,
        lam_slider.value, phi_dropdown.value,
    )
    with fig.batch_update():
        fig.data[0].x = kk
        fig.data[0].y = TT
        fig.data[0].z = iv

    update_arb_label(arb_free, max_cond)


# wire all widgets to update
for w in [rho_slider, sigma_slider, eta_slider, gamma_slider, lam_slider, phi_dropdown]:
    w.observe(on_change, names="value")

# --- show/hide eta/gamma vs lam based on phi model ---
def on_phi_change(change):
    is_heston = phi_dropdown.value == "heston_like"
    lam_slider.layout.display   = ""      if is_heston else "none"
    eta_slider.layout.display   = "none"  if is_heston else ""
    gamma_slider.layout.display = "none"  if is_heston else ""
    on_change(change)

phi_dropdown.observe(on_phi_change, names="value")

# trigger initial visibility
on_phi_change(None)

# --- layout ---
controls = widgets.VBox([
    phi_dropdown,
    rho_slider,
    sigma_slider,
    lam_slider,
    eta_slider,
    gamma_slider,
    arb_label,
])

display(widgets.HBox([controls, fig]))

download_btn = widgets.Button(
    description="Download CSV",
    button_style="primary",
    icon="download",
    layout=widgets.Layout(width="150px", margin="10px 0px 0px 0px"),
)
download_status = widgets.HTML(value="")


def download_csv(b):
    """Build CSV from current slider values and trigger browser download."""
    
    # get current parameters
    rho       = rho_slider.value
    sigma_atm = sigma_slider.value
    eta       = eta_slider.value
    gamma     = gamma_slider.value
    lam       = lam_slider.value
    phi_model = phi_dropdown.value

    # rebuild surface at higher resolution for the CSV
    kk, TT, iv, arb_free, max_cond = make_surface(
        rho, sigma_atm, eta, gamma, lam, phi_model
    )

    # also compute total variance w
    phi_obj = (
        HestonLikePhi(lam=lam)                     if phi_model == "heston_like"        else
        PowerLawPhi(eta=eta, gamma=gamma)           if phi_model == "power_law"          else
        StabilizedPowerLawPhi(eta=eta, gamma=gamma)
    )
    model      = SSVIFormula(rho=rho, phi=phi_obj)
    theta_func = lambda T: SSVIFormula.theta_of_T(T, sigma_atm=sigma_atm)
    theta_TT   = theta_func(TT)
    w          = model.evaluate(kk, theta_TT)

    # flatten to long format
    df_out = pd.DataFrame({
        "log_moneyness":  kk.ravel(),
        "maturity":       TT.ravel(),
        "theta":          theta_TT.ravel(),
        "total_variance": w.ravel(),
        "implied_vol":    iv.ravel(),
    })

    # add parameter metadata as columns so the file is self-documenting
    df_out["param_phi_model"] = phi_model
    df_out["param_rho"]       = rho
    df_out["param_sigma_atm"] = sigma_atm
    df_out["param_eta"]       = eta       if phi_model != "heston_like" else np.nan
    df_out["param_gamma"]     = gamma     if phi_model != "heston_like" else np.nan
    df_out["param_lam"]       = lam       if phi_model == "heston_like" else np.nan
    df_out["arb_free"]        = arb_free

    # build filename from parameters
    if phi_model == "heston_like":
        param_str = f"lam{lam:.1f}"
    else:
        param_str = f"eta{eta:.2f}_gamma{gamma:.2f}"

    filename = f"ssvi_{phi_model}_rho{rho:.2f}_sigma{sigma_atm:.2f}_{param_str}.csv"

    # encode to base64 and trigger browser download
    csv_str  = df_out.to_csv(index=False)
    b64      = base64.b64encode(csv_str.encode()).decode()

    js_code = f"""
    var link = document.createElement('a');
    link.href = 'data:text/csv;base64,{b64}';
    link.download = '{filename}';
    document.body.appendChild(link);
    link.click();
    document.body.removeChild(link);
    """
    display(Javascript(js_code))

    download_status.value = (
        f"<span style='color:green'>✓ Downloaded <b>{filename}</b> "
        f"— {len(df_out):,} rows</span>"
    )


download_btn.on_click(download_csv)

# --- updated layout with download button ---
controls = widgets.VBox([
    phi_dropdown,
    rho_slider,
    sigma_slider,
    lam_slider,
    eta_slider,
    gamma_slider,
    arb_label,
    download_btn,
    download_status,
])